# CLiP-SSM: Clinically-constrained, Leakage-penalized, switching State-Space generator
**Goal:** beat DG-HMM (Momenzadeh & Oshaghi 2026) on synthetic ICU time-series, using the same kind of metrics, and fix its documented weaknesses.

**What this notebook does**
1. Loads one dataset (set `DATASET` to `mimic3`, `physionet2012` or `physionet2019`; run the notebook once per dataset): hourly, 48 h, 8 continuous + 1 binary feature (MIMIC: ventilation; PhysioNet-2012: MechVent; PhysioNet-2019: SepsisLabel). If the data is not attached it falls back to a *simulated* demo set (pipeline test only).
2. Trains 9 baselines (HMM, RGAN, Recurrent VAE, **TimeGAN, TimeVAE, MedGAN-lite, TimeDiff-lite, DiffusionTS-lite**, **DG-HMM re-implementation**) and our **CLiP-SSM** under identical data/metric code. The *-lite* models are compact re-implementations of the cited designs, not the official code.
3. Evaluates fidelity, temporal consistency, clinical validity, privacy (incl. subgroup MIA) and utility (TSTR).

**Kaggle setup:** Add your MIMIC-III v1.4 CSVs as a *private* dataset (requires your own PhysioNet credentialed access; check the DUA before uploading). Needed files: ICUSTAYS, PATIENTS, CHARTEVENTS, LABEVENTS. Turn on GPU + Internet (for `hmmlearn`).

**Novel pieces (vs DG-HMM)**
- **Duration-aware switching prior**: state transitions conditioned on dwell time and the continuous latent (non-geometric durations; DG-HMM's HMM is first-order, geometric).
- **Constraint-by-construction decoder + rule-in-the-loop**: bounded decoder, differentiable clinical-rule penalties applied on *prior rollouts* (the actual generation path).
- **Leakage-aware training**: outlier-adaptive nearest-record margin, calibrated on held-out data, applied to rollouts (targets the subgroup/rare-trajectory MIA risk the paper left untested).

Caveat: NHANES/NESARC are not genuinely sequential, so the temporal claims are benchmarked on MIMIC-III only here.

In [ ]:
import os, sys, time, math, json, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
try:
    import hmmlearn
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hmmlearn"])
import torch, torch.nn as nn, torch.nn.functional as F
from scipy import stats
from sklearn.metrics import roc_auc_score

SMOKE = os.environ.get("SMOKE") == "1"
CFG = dict(SEED=0, T=48, N_MAX=15000, EP_CLIP=60, EP_BASE=200, BATCH=128, N_SYN=5000,
           LAM_RULE=50.0, LAM_PRIV=20.0, LAM_MMD=30.0, LAM_ENT=0.5, LAM_DYN=30.0, LAM_MARG=10.0, LAM_SW=0.5, LAM_COR=20.0, LAM_CW=10.0, TUNE_BASE=True, TUNE=True, ABL_SEEDS=[0, 1], RUN_ABLATIONS=True, RUN_SWEEP=False, SEEDS=[0, 1, 2, 3, 4])  # v5 first pass; final: SEEDS=[0,1,2,3,4], RUN_ABLATIONS=True  # first pass: 1 seed, no ablations. Final run: SEEDS=[0,1,2,3,4], RUN_ABLATIONS=True
if SMOKE:
    CFG.update(EP_CLIP=2, EP_BASE=2, N_SYN=300, N_MAX=600, SEEDS=[0], RUN_SWEEP=True, RUN_ABLATIONS=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
T = CFG["T"]
SIG_FLOOR = 0.005   # decoder noise floor in unit space (0.02 in v1-v3 forced noise on near-constant variables)
FEATS = [("hr", 30, 220), ("sbp", 50, 250), ("dbp", 20, 150), ("rr", 4, 60), ("spo2", 50, 100),
         ("temp", 34, 42), ("glucose", 40, 600), ("lactate", 0.2, 20), ("vent", 0, 1)]
D = len(FEATS)
LO = np.array([f[1] for f in FEATS], np.float32); HI = np.array([f[2] for f in FEATS], np.float32)
IDX = {f[0]: i for i, f in enumerate(FEATS)}
LO_t = torch.tensor(LO, device=DEV); HI_t = torch.tensor(HI, device=DEV)
print("device:", DEV)

## 1. Data (all features stored in [0,1] "unit space" = (x-lo)/(hi-lo) using physiological bounds)

In [ ]:
CH = {"hr": [211, 220045], "sbp": [51, 455, 220050, 220179], "dbp": [8368, 8441, 220051, 220180],
      "rr": [618, 615, 220210, 224690], "spo2": [646, 220277], "temp": [223762, 676], "tempF": [223761, 678],
      "glucose": [807, 811, 1529, 220621, 226537], "vent": [720, 223848, 223849]}
ITEM2F = {i: k for k, v in CH.items() for i in v}
LAB2F = {50809: "glucose", 50931: "glucose", 50813: "lactate"}

def find_mimic():
    for root, _, files in os.walk("/kaggle/input"):
        up = [f.upper() for f in files]
        if "CHARTEVENTS.CSV" in up or "CHARTEVENTS.CSV.GZ" in up:
            return root
    return None

def ffill(X):
    n, t, d = X.shape
    idx = np.where(~np.isnan(X), np.arange(t)[None, :, None], 0)
    np.maximum.accumulate(idx, axis=1, out=idx)
    return np.take_along_axis(X, idx, axis=1)

def finalize(X, vent_mode):
    """X [N,T,D] raw units with NaN. Drops artefacts, fills, converts to unit space. Last feature is the binary one."""
    X = X.copy(); X[(X < LO) | (X > HI)] = np.nan
    X = X[(~np.isnan(X[:, :, IDX["hr"]])).sum(1) >= 24]
    if vent_mode == "dilate":
        v = np.nan_to_num(X[:, :, -1]); v0 = v.copy()
        for lag in range(1, 12): v[:, lag:] = np.maximum(v[:, lag:], v0[:, :-lag])
    else:
        v = np.nan_to_num(ffill(X[:, :, -1:].copy())[:, :, 0])
    X = ffill(X); X = ffill(X[:, ::-1])[:, ::-1]
    for d in range(D - 1):
        med = np.nanmedian(X[:, :, d]); med = (LO[d] + HI[d]) / 2 if np.isnan(med) else med
        X[:, :, d] = np.where(np.isnan(X[:, :, d]), med, X[:, :, d])
    X[:, :, -1] = v
    return np.clip((X - LO) / (HI - LO), 0, 1).astype(np.float32)

def build_physionet2012():
    """PhysioNet/CinC 2012 (open access): set-a/b/c patient .txt files (Time,Parameter,Value). SpO2 slot <- SaO2, binary slot <- MechVent."""
    M = {"HR": "hr", "SysABP": "sbp", "NISysABP": "sbp", "DiasABP": "dbp", "NIDiasABP": "dbp", "RespRate": "rr", "SaO2": "spo2",
         "Temp": "temp", "Glucose": "glucose", "Lactate": "lactate", "MechVent": "vent"}
    fs = [os.path.join(r, f) for r, _, ff in os.walk("/kaggle/input") for f in ff
          if f.endswith(".txt") and not f.lower().startswith(("outcomes", "readme"))]
    np.random.default_rng(0).shuffle(fs); X = []
    for fp in fs:
        try: df = pd.read_csv(fp)
        except Exception: continue
        if list(df.columns[:3]) != ["Time", "Parameter", "Value"]: continue
        df = df[df.Parameter.isin(M)]
        if df.empty: continue
        df = df.assign(H=df.Time.astype(str).str.split(":").str[0].astype(int), F=df.Parameter.map(M)); df = df[df.H < T]
        a = np.full((T, D), np.nan, np.float32)
        for (hh, ff), v in df.groupby(["H", "F"]).Value.mean().items(): a[hh, IDX[ff]] = v
        X.append(a)
        if len(X) >= CFG["N_MAX"] * 1.3: break
    U = finalize(np.stack(X), "dilate")[:CFG["N_MAX"]]; print("PhysioNet2012 cohort:", U.shape); return U

def build_physionet2019():
    """PhysioNet/CinC 2019 sepsis (open access): hourly .psv files. SpO2 slot <- O2Sat, binary slot <- SepsisLabel."""
    M = {"HR": "hr", "SBP": "sbp", "DBP": "dbp", "Resp": "rr", "O2Sat": "spo2", "Temp": "temp", "Glucose": "glucose", "Lactate": "lactate", "SepsisLabel": "vent"}
    fs = [os.path.join(r, f) for r, _, ff in os.walk("/kaggle/input") for f in ff if f.endswith(".psv")]
    np.random.default_rng(0).shuffle(fs); X = []
    for fp in fs:
        df = pd.read_csv(fp, sep="|", usecols=lambda c: c in M)
        if len(df) < T: continue
        a = np.full((T, D), np.nan, np.float32)
        for c, f in M.items():
            if c in df: a[:, IDX[f]] = df[c].values[:T]
        X.append(a)
        if len(X) >= CFG["N_MAX"] * 1.3: break
    U = finalize(np.stack(X), "ffill")[:CFG["N_MAX"]]; print("PhysioNet2019 cohort:", U.shape); return U

def build_mimic(root):
    def P(n):
        return next(os.path.join(root, f) for f in os.listdir(root) if f.upper() in (n + ".CSV", n + ".CSV.GZ"))
    def rd(n, cols, **kw):
        df = pd.read_csv(P(n), usecols=lambda c: c.upper() in cols, **kw)
        df.columns = df.columns.str.upper(); return df
    icu = rd("ICUSTAYS", {"SUBJECT_ID", "HADM_ID", "ICUSTAY_ID", "INTIME", "LOS"})
    pat = rd("PATIENTS", {"SUBJECT_ID", "DOB"})
    pat["DOBY"] = pat["DOB"].astype(str).str[:4].astype(int)
    icu["INTIME"] = pd.to_datetime(icu["INTIME"])
    icu = icu.merge(pat[["SUBJECT_ID", "DOBY"]], on="SUBJECT_ID")
    icu = icu[(icu.INTIME.dt.year - icu.DOBY >= 18) & (icu.LOS >= 2.0)]
    icu = icu.sort_values("INTIME").drop_duplicates("SUBJECT_ID")
    if len(icu) > CFG["N_MAX"]:
        icu = icu.sample(CFG["N_MAX"], random_state=CFG["SEED"])
    stays = icu.ICUSTAY_ID.values; sid = {s: i for i, s in enumerate(stays)}
    intime = dict(zip(icu.ICUSTAY_ID, icu.INTIME)); hadm2s = dict(zip(icu.HADM_ID, icu.ICUSTAY_ID))
    rows = []
    t0 = time.time()
    for ch in pd.read_csv(P("CHARTEVENTS"), usecols=lambda c: c.upper() in {"ICUSTAY_ID", "ITEMID", "CHARTTIME", "VALUENUM"},
                          chunksize=4_000_000):
        ch.columns = ch.columns.str.upper()
        ch = ch[ch.ICUSTAY_ID.isin(sid) & ch.ITEMID.isin(ITEM2F)].copy()
        if ch.empty: continue
        ch["F"] = ch.ITEMID.map(ITEM2F)
        ch.loc[ch.F == "vent", "VALUENUM"] = 1.0
        ch = ch.dropna(subset=["VALUENUM"])
        ch.loc[ch.F == "tempF", "VALUENUM"] = (ch.loc[ch.F == "tempF", "VALUENUM"] - 32) * 5 / 9
        ch.loc[ch.F == "tempF", "F"] = "temp"
        ch["H"] = ((pd.to_datetime(ch.CHARTTIME) - ch.ICUSTAY_ID.map(intime)).dt.total_seconds() // 3600)
        ch = ch[(ch.H >= 0) & (ch.H < T)]
        rows.append(ch.groupby(["ICUSTAY_ID", "F", "H"]).VALUENUM.mean().reset_index())
    print("chartevents done %.0fs" % (time.time() - t0))
    for lb in pd.read_csv(P("LABEVENTS"), usecols=lambda c: c.upper() in {"HADM_ID", "ITEMID", "CHARTTIME", "VALUENUM"},
                          chunksize=4_000_000):
        lb.columns = lb.columns.str.upper()
        lb = lb[lb.HADM_ID.isin(hadm2s) & lb.ITEMID.isin(LAB2F)].dropna(subset=["VALUENUM"]).copy()
        if lb.empty: continue
        lb["ICUSTAY_ID"] = lb.HADM_ID.map(hadm2s); lb["F"] = lb.ITEMID.map(LAB2F)
        h = ((pd.to_datetime(lb.CHARTTIME) - lb.ICUSTAY_ID.map(intime)).dt.total_seconds() // 3600)
        lb["H"] = np.where((h < 0) & (h >= -24), 0, h)
        lb = lb[(lb.H >= 0) & (lb.H < T)]
        rows.append(lb.groupby(["ICUSTAY_ID", "F", "H"]).VALUENUM.mean().reset_index())
    agg = pd.concat(rows).groupby(["ICUSTAY_ID", "F", "H"]).VALUENUM.mean().reset_index()
    X = np.full((len(stays), T, D), np.nan, np.float32)
    fi = agg.F.map(IDX).values
    ok = (agg.VALUENUM.values >= LO[fi]) & (agg.VALUENUM.values <= HI[fi])  # drop charting artefacts
    agg, fi = agg[ok], fi[ok]
    X[agg.ICUSTAY_ID.map(sid).values, agg.H.astype(int).values, fi] = agg.VALUENUM.values
    U = finalize(X, "dilate"); print("MIMIC cohort:", U.shape); return U

def make_demo(n, K=4, seed=0):
    r = np.random.default_rng(seed); means = r.uniform(.25, .75, (K, D)); U = np.zeros((n, T, D), np.float32)
    for i in range(n):
        t, s = 0, r.integers(K); noise = np.zeros(D)
        while t < T:
            L = 4 + r.poisson(6)
            for _ in range(min(L, T - t)):
                noise = .8 * noise + .03 * r.standard_normal(D); U[i, t] = means[s] + noise; t += 1
            s = r.integers(K)
    U[:, :, -1] = (r.random((n, T)) < U[:, :, -1]).astype(np.float32)
    U[:, :, IDX["sbp"]] = np.clip(U[:, :, IDX["dbp"]] + .2, 0, 1)
    return np.clip(U, 0, 1)

DATASET = os.environ.get("DATASET", "physionet2012")   # "mimic3" | "physionet2012" | "physionet2019"  (run the notebook once per dataset)
U_all = None
try:
    if DATASET == "mimic3":
        root = find_mimic(); U_all = build_mimic(root) if root else None
    elif DATASET == "physionet2012": U_all = build_physionet2012()
    elif DATASET == "physionet2019": U_all = build_physionet2019()
except Exception as e:
    print("loader failed:", repr(e)); U_all = None
if U_all is None or len(U_all) < 200:
    print("!!! %s not found/usable -> SIMULATED demo data. Pipeline test only. !!!" % DATASET)
    U_all = make_demo(CFG["N_MAX"] if not SMOKE else 600); MODE = "demo"
else: MODE = DATASET
rng = np.random.default_rng(CFG["SEED"]); perm = rng.permutation(len(U_all))
n1, n2 = int(.7 * len(perm)), int(.85 * len(perm))
SPL = dict(tr=U_all[perm[:n1]], va=U_all[perm[n1:n2]], te=U_all[perm[n2:]])
print({k: v.shape for k, v in SPL.items()})

## 2. Clinical rules, metrics (shared by all methods)

In [ ]:
def rule_hinge(u):
    """u [B,T,D] unit space (torch). Returns normalised violations [B,T,6]; 0 == rule satisfied."""
    x = LO_t + u * (HI_t - LO_t); g = lambda k: x[..., IDX[k]]
    pp = g("sbp") - g("dbp")
    def rate(v, lim):
        d = (v[:, 1:] - v[:, :-1]).abs(); return F.pad(F.relu(d - lim) / lim, (1, 0))
    return torch.stack([F.relu(5 - pp) / 50, F.relu(pp - 130) / 50, rate(g("hr"), 60), rate(g("spo2"), 15),
                        rate(g("sbp"), 80), rate(g("temp"), 1.5)], -1)

def flat(U): return U.reshape(len(U), -1)

def min_dist(A, B, bs=512):
    At = torch.tensor(flat(A), device=DEV); Bt = torch.tensor(flat(B), device=DEV); out = []
    for i in range(0, len(At), bs):
        out.append((torch.cdist(At[i:i + bs], Bt) / math.sqrt(At.shape[1])).min(1).values.cpu())
    return torch.cat(out).numpy()

def dtw_nn(S, R):
    a, b = len(S), len(R); St = torch.tensor(S); Rt = torch.tensor(R)
    C = torch.cdist(St.reshape(a * T, D), Rt.reshape(b * T, D)).reshape(a, T, b, T).permute(0, 2, 1, 3).reshape(a * b, T, T)
    acc = torch.full((a * b, T + 1, T + 1), float("inf")); acc[:, 0, 0] = 0
    for i in range(1, T + 1):
        for j in range(1, T + 1):
            acc[:, i, j] = C[:, i - 1, j - 1] + torch.minimum(torch.minimum(acc[:, i - 1, j], acc[:, i, j - 1]), acc[:, i - 1, j - 1])
    return (acc[:, T, T].reshape(a, b).min(1).values / (2 * T)).mean().item()

def acf_lag(U, lag=10):
    x = U - U.mean(1, keepdims=True); num = (x[:, :-lag] * x[:, lag:]).sum(1); den = (x ** 2).sum(1)
    with np.errstate(all="ignore"): return np.nanmean(num / den, axis=0)  # per feature

def groups(U):
    sbp = LO[IDX["sbp"]] + U[:, 24:, IDX["sbp"]] * (HI[IDX["sbp"]] - LO[IDX["sbp"]])
    hypo = (sbp < 90).any(1); vent = U[:, :, IDX["vent"]].mean(1) > 0.5
    return {"hypo": hypo, "no_hypo": ~hypo, "vent": vent, "no_vent": ~vent}

def evaluate(S_raw, split=None, name=""):
    split = split or SPL; R, Rtr = split["te"], split["tr"]
    bounds = float(((S_raw >= 0) & (S_raw <= 1)).mean() * 100)
    S = np.clip(S_raw, 0, 1).astype(np.float32); S[..., -1] = (S[..., -1] > .5)   # baselines get post-hoc clip + binarise
    rs = np.random.default_rng(1); sub = lambda A: A.reshape(-1, D)[rs.choice(len(A) * T, min(100000, len(A) * T), replace=False)]
    Rp, Sp = sub(R), sub(S)
    out = {"model": name}
    ksf = [stats.ks_2samp(Rp[:, d], Sp[:, d]).statistic for d in range(D)]; out["KS"] = np.mean(ksf)
    for d in range(D): out["KS_" + FEATS[d][0]] = ksf[d]
    Cr, Cs = np.nan_to_num(np.corrcoef(R.reshape(-1, D).T)), np.nan_to_num(np.corrcoef(S.reshape(-1, D).T))
    out["CorrPres"] = 1 - np.abs(Cr - Cs)[np.triu_indices(D, 1)].mean()
    ar, as_ = acf_lag(R), acf_lag(S)
    out["ACF10"] = np.nanmean(as_); out["ACF10_err"] = np.nanmean(np.abs(ar - as_))
    out["DTW"] = dtw_nn(S[:100], R[:200])
    out["ACFcurve_err"] = np.mean([abs(np.nanmean(acf_lag(R, l)) - np.nanmean(acf_lag(S, l))) for l in range(1, 25)])
    out["SkewErr"] = np.nanmean(np.abs(stats.skew(Rp, 0) - stats.skew(Sp, 0)))
    out["KurtErr"] = np.nanmean(np.abs(stats.kurtosis(Rp, 0) - stats.kurtosis(Sp, 0)))
    v = rule_hinge(torch.tensor(S, device=DEV)).max(-1).values.cpu().numpy() <= 1e-4
    out["RuleRec%"] = v.all(1).mean() * 100; out["RuleStep%"] = v.mean() * 100; out["Bounds%_raw"] = bounds
    vr = rule_hinge(torch.tensor(R, device=DEV)).max(-1).values.cpu().numpy() <= 1e-4
    out["RuleReal%"] = vr.all(1).mean() * 100; out["RuleGap"] = abs(out["RuleRec%"] - out["RuleReal%"])   # distance to the real-data rate
    n = min(2000, len(R), len(Rtr)); mem, non = Rtr[:n], R[:n]
    sm, sn = -min_dist(mem, S), -min_dist(non, S)
    auc = roc_auc_score(np.r_[np.ones(n), np.zeros(n)], np.r_[sm, sn]); out["MIA_AUC"] = auc
    out["MIA_resist%"] = 100 * (1 - 2 * abs(auc - .5))
    gm, gn = groups(mem), groups(non); w = []
    for k in gm:
        if gm[k].sum() > 20 and gn[k].sum() > 20:
            w.append(roc_auc_score(np.r_[np.ones(gm[k].sum()), np.zeros(gn[k].sum())], np.r_[sm[gm[k]], sn[gn[k]]]))
    out["MIA_worstgrp_AUC"] = max(w) if w else np.nan
    ref = split["va"][:n]; rm, rn = min_dist(mem, ref), min_dist(non, ref)          # calibrated attack: distance to synthetic vs. to a disjoint real reference
    cal = np.r_[rm + sm, rn + sn]; cauc = roc_auc_score(np.r_[np.ones(n), np.zeros(n)], cal); out["MIA_cal_AUC"] = cauc; out["MIA_cal_dev"] = abs(cauc - .5)
    ds = min_dist(S[:2000], Rtr); dt = min_dist(R[:2000], Rtr)
    out["DCR_p5_ratio"] = np.percentile(ds, 5) / np.percentile(dt, 5)   # >=1: synthetic no closer to train than test is
    return out

# utility (TSTR): from hours 0-23 predict (a) mean of hours 24-47 for continuous features, (b) any SBP<90 in 24-47
def mk_task(U):
    X = U[:, :24]; Y = U[:, 24:, :D - 1].mean(1); return X, Y, groups(U)["hypo"].astype(np.float32)
class Pred(nn.Module):
    def __init__(s): super().__init__(); s.g = nn.GRU(D, 64, batch_first=True); s.o = nn.Linear(64, D)
    def forward(s, x): h, _ = s.g(x); return s.o(h[:, -1])
def fit_pred(Utrain, Ute, ep=20):
    X, Y, L = [torch.tensor(a, device=DEV) for a in mk_task(Utrain)]
    Xt, Yt, Lt = [torch.tensor(a, device=DEV) for a in mk_task(Ute)]
    m = Pred().to(DEV); opt = torch.optim.Adam(m.parameters(), 3e-3)
    for _ in range(ep):
        p = torch.randperm(len(X), device=DEV)
        for i in range(0, len(X), 256):
            b = p[i:i + 256]; o = m(X[b])
            loss = 10 * F.mse_loss(o[:, :-1], Y[b]) + F.binary_cross_entropy_with_logits(o[:, -1], L[b])
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): o = m(Xt)
    mse = F.mse_loss(o[:, :-1], Yt).item()
    try: auc = roc_auc_score(Lt.cpu().numpy(), o[:, -1].cpu().numpy())
    except Exception: auc = float("nan")
    return mse, auc
TRTR = fit_pred(SPL["tr"], SPL["te"]); print("TRTR (mse, auc):", TRTR)
def utility(S):
    S = np.clip(S, 0, 1).astype(np.float32); S[..., -1] = (S[..., -1] > .5)
    mse, auc = fit_pred(S[:len(SPL["tr"])], SPL["te"])
    return {"Util_fore%": 100 * TRTR[0] / mse, "Util_auc%": 100 * auc / TRTR[1]}
real_ref = evaluate(SPL["va"][:len(SPL["te"])].copy(), name="REAL(val) reference"); print(pd.Series(real_ref))

## 3. Baselines (re-implemented under identical data and metrics)

In [ ]:
def fit_generic(model, loss_fn, U, epochs, lr=2e-3, bs=None):
    bs = bs or CFG["BATCH"]; X = torch.tensor(U, device=DEV); opt = torch.optim.Adam(model.parameters(), lr)
    for ep in range(epochs):
        p = torch.randperm(len(X), device=DEV); tot = 0
        for i in range(0, len(X) - bs + 1, bs):
            loss = loss_fn(X[p[i:i + bs]], ep / epochs); opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.); opt.step(); tot += loss.item()
        if ep % max(1, epochs // 5) == 0: print("  ep", ep, "loss %.4f" % (tot / max(1, len(X) // bs)))
    return model.eval()

def batched(fn, n, bs=1000):
    with torch.no_grad(): return np.concatenate([fn(min(bs, n - i)).cpu().numpy() for i in range(0, n, bs)])

class FlatVAE(nn.Module):
    def __init__(s, L=64, H=512):
        super().__init__(); s.L = L
        s.e = nn.Sequential(nn.Linear(T * D, H), nn.ReLU(), nn.Linear(H, 2 * L)); s.d = nn.Sequential(nn.Linear(L, H), nn.ReLU(), nn.Linear(H, T * D))
    def loss(s, x, f):
        mu, lv = s.e(x.reshape(len(x), -1)).chunk(2, -1); z = mu + torch.randn_like(mu) * (.5 * lv).exp()
        rec = ((s.d(z) - x.reshape(len(x), -1)) ** 2).sum(-1).mean()
        return rec + min(1, f / .3) * (-.5 * (1 + lv - mu ** 2 - lv.exp()).sum(-1)).mean() * .05
    def sample(s, n): return s.d(torch.randn(n, s.L, device=DEV)).reshape(n, T, D)

class RecVAE(nn.Module):
    def __init__(s, L=32, H=128):
        super().__init__(); s.L = L; s.g = nn.GRU(D, H, batch_first=True); s.m = nn.Linear(H, 2 * L)
        s.dg = nn.GRU(L, H, batch_first=True); s.o = nn.Linear(H, D)
    def dec(s, z): h, _ = s.dg(z.unsqueeze(1).repeat(1, T, 1)); return s.o(h)
    def loss(s, x, f):
        _, h = s.g(x); mu, lv = s.m(h[0]).chunk(2, -1); z = mu + torch.randn_like(mu) * (.5 * lv).exp()
        return ((s.dec(z) - x) ** 2).sum(-1).sum(-1).mean() / T + min(1, f / .3) * (-.5 * (1 + lv - mu ** 2 - lv.exp()).sum(-1)).mean() * .05
    def sample(s, n): return s.dec(torch.randn(n, s.L, device=DEV))

def fit_rgan(U, epochs, H=128, Zn=8, bs=128):
    G = nn.LSTM(Zn, H, batch_first=True).to(DEV); Go = nn.Linear(H, D).to(DEV)
    Dn = nn.LSTM(D, H, batch_first=True).to(DEV); Do = nn.Linear(H, 1).to(DEV)
    oG = torch.optim.Adam(list(G.parameters()) + list(Go.parameters()), 1e-3, betas=(.5, .999))
    oD = torch.optim.Adam(list(Dn.parameters()) + list(Do.parameters()), 1e-3, betas=(.5, .999))
    gen = lambda n: Go(G(torch.randn(n, T, Zn, device=DEV))[0]); dis = lambda x: Do(Dn(x)[0]).squeeze(-1)
    X = torch.tensor(U, device=DEV); bce = F.binary_cross_entropy_with_logits
    for ep in range(epochs):
        p = torch.randperm(len(X), device=DEV)
        for i in range(0, len(X) - bs + 1, bs):
            xr = X[p[i:i + bs]]; xf = gen(bs)
            lD = bce(dis(xr), torch.ones(bs, T, device=DEV)) + bce(dis(xf.detach()), torch.zeros(bs, T, device=DEV))
            oD.zero_grad(); lD.backward(); oD.step()
            lG = bce(dis(gen(bs)), torch.ones(bs, T, device=DEV)); oG.zero_grad(); lG.backward(); oG.step()
    return lambda n: gen(n)

class DGHMM(nn.Module):
    """Re-implementation from the paper text (encoder -> HMM on latent -> state-conditioned decoder)."""
    def __init__(s, K=6, Z=16, H=128):
        super().__init__(); s.K, s.Z = K, Z
        s.enc = nn.Sequential(nn.Linear(D, H), nn.ReLU(), nn.Linear(H, H), nn.ReLU()); s.mu = nn.Linear(H, Z); s.lv = nn.Linear(H, Z)
        s.bn = nn.BatchNorm1d(Z); s.drop = nn.Dropout(0.1)
        s.dec = nn.Sequential(nn.Linear(Z + K, H), nn.ReLU(), nn.Linear(H, H), nn.ReLU(), nn.Linear(H, D))
        s.pi = nn.Parameter(torch.zeros(K)); s.A = nn.Parameter(torch.zeros(K, K)); s.m = nn.Parameter(torch.randn(K, Z)); s.l = nn.Parameter(torch.zeros(K, Z))
    def fwd(s, z):
        lp = F.log_softmax(s.A, -1); lv = s.l.clamp(-6, 3)
        lb = -.5 * (((z.unsqueeze(2) - s.m) ** 2) / lv.exp() + lv + math.log(2 * math.pi)).sum(-1)
        a = F.log_softmax(s.pi, 0) + lb[:, 0]; gam = [F.softmax(a, -1)]
        for t in range(1, T):
            a = torch.logsumexp(a.unsqueeze(2) + lp, 1) + lb[:, t]; gam.append(F.softmax(a, -1))
        return -torch.logsumexp(a, -1), torch.stack(gam, 1)
    def loss(s, x, f):
        h = s.enc(x); mu, lv = s.mu(h), s.lv(h).clamp(-8, 4); z = mu + torch.randn_like(mu) * (.5 * lv).exp()
        z = s.drop(s.bn(z.reshape(-1, s.Z)).reshape(len(x), T, s.Z)); nll, gam = s.fwd(z)
        rec = ((s.dec(torch.cat([z, gam], -1)) - x) ** 2).sum(-1).mean()
        return rec * 10 + 0.01 * nll.mean() / T + 1e-3 * (-.5 * (1 + lv - mu ** 2 - lv.exp()).sum(-1)).mean()
    def sample(s, n):
        st = torch.distributions.Categorical(logits=s.pi).sample((n,)); S = [st]
        for t in range(1, T): st = torch.distributions.Categorical(logits=s.A[st]).sample(); S.append(st)
        S = torch.stack(S, 1); z = s.m[S] + torch.randn(n, T, s.Z, device=DEV) * (.5 * s.l[S].clamp(-6, 3)).exp()
        return s.dec(torch.cat([z, F.one_hot(S, s.K).float()], -1))

In [ ]:
def temb(t, dim):
    half = dim // 2; f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half); a = t.float()[:, None] * f[None]
    return torch.cat([a.sin(), a.cos()], -1)
class BRNNNet(nn.Module):      # TimeDiff-style denoiser (bidirectional RNN)
    def __init__(s, H=96):
        super().__init__(); s.H = H; s.i = nn.Linear(D, H); s.t = nn.Sequential(nn.Linear(H, H), nn.SiLU(), nn.Linear(H, H))
        s.r = nn.GRU(H, H, 2, batch_first=True, bidirectional=True); s.o = nn.Linear(2 * H, D)
    def forward(s, x, t): return s.o(s.r(s.i(x) + s.t(temb(t, s.H)).unsqueeze(1))[0])
class TFNet(nn.Module):        # Diffusion-TS-style denoiser (transformer)
    def __init__(s, H=96):
        super().__init__(); s.H = H; s.i = nn.Linear(D, H); s.t = nn.Sequential(nn.Linear(H, H), nn.SiLU(), nn.Linear(H, H))
        s.pos = nn.Parameter(torch.zeros(1, T, H)); s.tf = nn.TransformerEncoder(nn.TransformerEncoderLayer(H, 4, 2 * H, 0.0, batch_first=True, norm_first=True), 3); s.o = nn.Linear(H, D)
    def forward(s, x, t): return s.o(s.tf(s.i(x) + s.pos + s.t(temb(t, s.H)).unsqueeze(1)))
class Diff(nn.Module):
    """Gaussian DDPM (cosine schedule, eps-prediction) on [-1,1]-scaled sequences; binary feature diffused as a real value and thresholded."""
    def __init__(s, net, NT=100):
        super().__init__(); s.net, s.NT = net, NT
        ab = torch.cos((torch.arange(NT + 1) / NT + .008) / 1.008 * math.pi / 2) ** 2; ab = ab / ab[0]
        beta = (1 - ab[1:] / ab[:-1]).clamp(1e-4, .999); s.register_buffer("beta", beta); s.register_buffer("alpha", 1 - beta); s.register_buffer("ab", torch.cumprod(1 - beta, 0))
    def loss(s, x, f):
        x0 = 2 * x - 1; t = torch.randint(0, s.NT, (len(x),), device=x.device); e = torch.randn_like(x0); ab = s.ab[t][:, None, None]
        return F.mse_loss(s.net(ab.sqrt() * x0 + (1 - ab).sqrt() * e, t), e)
    def sample(s, n):
        x = torch.randn(n, T, D, device=s.ab.device)
        for i in reversed(range(s.NT)):
            e = s.net(x, torch.full((n,), i, device=x.device)); x = (x - (1 - s.alpha[i]) / (1 - s.ab[i]).sqrt() * e) / s.alpha[i].sqrt()
            if i > 0: x = x + s.beta[i].sqrt() * torch.randn_like(x)
        return (x.clamp(-1, 1) + 1) / 2

class TimeVAE(nn.Module):      # base TimeVAE (conv encoder/decoder; no trend/seasonality blocks)
    def __init__(s, L=16):
        super().__init__(); s.L = L
        s.e = nn.Sequential(nn.Conv1d(D, 64, 3, 2, 1), nn.ReLU(), nn.Conv1d(64, 128, 3, 2, 1), nn.ReLU(), nn.Flatten()); s.m = nn.Linear(128 * (T // 4), 2 * L)
        s.fc = nn.Linear(L, 128 * (T // 4)); s.d = nn.Sequential(nn.ReLU(), nn.ConvTranspose1d(128, 64, 3, 2, 1, output_padding=1), nn.ReLU(), nn.ConvTranspose1d(64, D, 3, 2, 1, output_padding=1))
    def dec(s, z): return s.d(s.fc(z).reshape(len(z), 128, T // 4)).transpose(1, 2)
    def loss(s, x, f):
        mu, lv = s.m(s.e(x.transpose(1, 2))).chunk(2, -1); z = mu + torch.randn_like(mu) * (.5 * lv).exp()
        return 3 * ((s.dec(z) - x) ** 2).sum(-1).mean() + min(1, f / .3) * (-.5 * (1 + lv - mu ** 2 - lv.exp()).sum(-1)).mean() * .1
    def sample(s, n): return s.dec(torch.randn(n, s.L, device=DEV))

class MedGANlite(nn.Module):   # autoencoder + generator in code space, discriminator on decoded flattened sequence
    def __init__(s, L=128, Zn=64):
        super().__init__(); s.Zn = Zn
        s.enc = nn.Sequential(nn.Linear(T * D, 256), nn.ReLU(), nn.Linear(256, L), nn.Tanh()); s.dec = nn.Sequential(nn.Linear(L, 256), nn.ReLU(), nn.Linear(256, T * D))
        s.G = nn.Sequential(nn.Linear(Zn, L), nn.BatchNorm1d(L), nn.ReLU(), nn.Linear(L, L), nn.Tanh()); s.Dn = nn.Sequential(nn.Linear(T * D, 256), nn.LeakyReLU(.2), nn.Linear(256, 1))
    def sample(s, n): return s.dec(s.G(torch.randn(n, s.Zn, device=DEV))).reshape(n, T, D)
def fit_medgan(U, epochs, bs=128):
    m = MedGANlite().to(DEV); X = torch.tensor(U, device=DEV).reshape(len(U), -1); bce = F.binary_cross_entropy_with_logits
    oA = torch.optim.Adam(list(m.enc.parameters()) + list(m.dec.parameters()), 1e-3); oG = torch.optim.Adam(list(m.G.parameters()) + list(m.dec.parameters()), 1e-3, betas=(.5, .999)); oD = torch.optim.Adam(m.Dn.parameters(), 1e-3, betas=(.5, .999))
    for ep in range(max(1, epochs // 2)):
        p = torch.randperm(len(X), device=DEV)
        for i in range(0, len(X) - bs + 1, bs): l = F.mse_loss(m.dec(m.enc(X[p[i:i + bs]])), X[p[i:i + bs]]); oA.zero_grad(); l.backward(); oA.step()
    for ep in range(epochs):
        p = torch.randperm(len(X), device=DEV)
        for i in range(0, len(X) - bs + 1, bs):
            xr = X[p[i:i + bs]]; xf = m.dec(m.G(torch.randn(bs, m.Zn, device=DEV)))
            lD = bce(m.Dn(xr), torch.ones(bs, 1, device=DEV)) + bce(m.Dn(xf.detach()), torch.zeros(bs, 1, device=DEV)); oD.zero_grad(); lD.backward(); oD.step()
            lG = bce(m.Dn(xf), torch.ones(bs, 1, device=DEV)); oG.zero_grad(); lG.backward(); oG.step()
    return m.eval()

class RH(nn.Module):
    def __init__(s, i, o, act=None, l=2, H=64): super().__init__(); s.r = nn.GRU(i, H, l, batch_first=True); s.o = nn.Linear(H, o); s.a = act
    def forward(s, x): y = s.o(s.r(x)[0]); return torch.sigmoid(y) if s.a == "sig" else y
def fit_timegan(U, epochs, H=64, Zn=16, bs=128):
    """Compact TimeGAN (Yoon et al. 2019): embedder/recovery, generator, supervisor, discriminator; 3 training phases."""
    E, R, G, S, Dn = RH(D, H, "sig").to(DEV), RH(H, D, "sig").to(DEV), RH(Zn, H, "sig").to(DEV), RH(H, H, "sig", 1).to(DEV), RH(H, 1).to(DEV)
    opt = lambda ms, lr=1e-3: torch.optim.Adam([p for m in ms for p in m.parameters()], lr)
    oER, oGS, oD, oS = opt([E, R]), opt([G, S]), opt([Dn]), opt([S]); X = torch.tensor(U, device=DEV); bce = F.binary_cross_entropy_with_logits; ph = max(1, epochs // 2)
    def batches():
        p = torch.randperm(len(X), device=DEV)
        for i in range(0, len(X) - bs + 1, bs): yield X[p[i:i + bs]]
    for _ in range(ph):
        for x in batches(): l = F.mse_loss(R(E(x)), x); oER.zero_grad(); l.backward(); oER.step()
    for _ in range(ph):
        for x in batches(): h = E(x).detach(); l = F.mse_loss(S(h)[:, :-1], h[:, 1:]); oS.zero_grad(); l.backward(); oS.step()
    for _ in range(epochs):
        for x in batches():
            z = torch.randn(len(x), T, Zn, device=DEV); h = E(x); eh = G(z); hh = S(eh); xh = R(hh)
            gs = F.mse_loss(S(h.detach())[:, :-1], h.detach()[:, 1:])
            mom = (xh.var(0).add(1e-6).sqrt() - x.var(0).add(1e-6).sqrt()).abs().mean() + (xh.mean(0) - x.mean(0)).abs().mean()
            lG = bce(Dn(hh), torch.ones(len(x), T, 1, device=DEV)) + bce(Dn(eh), torch.ones(len(x), T, 1, device=DEV)) + 100 * gs.add(1e-8).sqrt() + 100 * mom
            oGS.zero_grad(); lG.backward(); oGS.step()
            h = E(x); lE = 10 * F.mse_loss(R(h), x).add(1e-8).sqrt() + 0.1 * F.mse_loss(S(h)[:, :-1], h[:, 1:]); oER.zero_grad(); lE.backward(); oER.step()
            with torch.no_grad(): hr = E(x); eh = G(z); hh = S(eh)
            lD = bce(Dn(hr), torch.ones(len(x), T, 1, device=DEV)) + bce(Dn(hh), torch.zeros(len(x), T, 1, device=DEV)) + bce(Dn(eh), torch.zeros(len(x), T, 1, device=DEV))
            oD.zero_grad(); lD.backward(); oD.step()
    return lambda n: R(S(G(torch.randn(n, T, Zn, device=DEV))))

## 4. CLiP-SSM (proposed)

In [ ]:
class CLiP(nn.Module):
    """Switching state-space generator: regime prior with dwell counter, residual (persistent) latent dynamics,
    sequence-level latent u for patient-level variation, bounded decoder."""
    def __init__(s, K=8, Z=24, U=16, H=128, use_dwell=True, use_u=True, use_hold=True):
        super().__init__(); s.K, s.Z, s.U, s.use_dwell, s.use_u, s.use_hold = K, Z, U, use_dwell, use_u, use_hold
        s.enc = nn.GRU(D, H, batch_first=True, bidirectional=True)
        mlp = lambda i, o: nn.Sequential(nn.Linear(i, H), nn.GELU(), nn.Linear(H, o))
        s.qu = nn.Linear(2 * H, 2 * U)
        s.qs, s.qz = mlp(2 * H + K + Z + U, K), mlp(2 * H + K + Z + U, 2 * Z)
        s.ps, s.pz = mlp(K + 1 + Z + U, K), mlp(Z + K + U, 2 * Z)
        s.dec = nn.Sequential(nn.Linear(Z + K + U, 2 * H), nn.GELU(), nn.Linear(2 * H, 2 * H), nn.GELU(), nn.Linear(2 * H, 3 * D))
        s.sm = nn.Parameter(torch.randn(K, Z) * 0.1)      # regime-specific latent drift, so a regime has a physiological meaning
        s.kappa = nn.Parameter(torch.tensor(2.0))         # self-transition bias, starts sticky
        s.rho_logit = nn.Parameter(torch.linspace(1.0, 5.5, Z))   # multi-timescale latent: per-dimension persistence from about 0.7 to 0.996 per hour   # mean-reverting latent: z_t = rho*z_{t-1} + increment
        s.pz[-1].bias.data[Z:].fill_(-4.0)          # start with small latent increments (persistent trajectories)
    def decode(s, z, st, u):
        if z.dim() == 3 and u.dim() == 2: u = u.unsqueeze(1).expand(-1, z.shape[1], -1)
        return s.dec(torch.cat([z, st, u], -1)).chunk(3, -1)   # value logit, value scale, hold/update gate logit
    def prior_s(s, sp, d, zp, u): return s.ps(torch.cat([sp, torch.log1p(d).unsqueeze(-1) * float(s.use_dwell), zp, u], -1)) + s.kappa * sp
    def prior_z(s, zp, st, u):
        m, lv = s.pz(torch.cat([zp, st, u], -1)).chunk(2, -1); return torch.sigmoid(s.rho_logit) * zp + m + st @ s.sm, lv.clamp(-8, 3)
    @staticmethod
    def sample_step(m, ls, hg, xprev, first, hard_out=True, grad=False, hold=True):
        """Hold-aware observation: with probability sigmoid(hg) a new value is measured, otherwise the previous value is held
        (EHR series are sparse and forward-filled, so many variables are exactly constant for long stretches)."""
        mu = torch.sigmoid(m[..., :-1]); sg = F.softplus(ls[..., :-1] - 3) + SIG_FLOOR
        c = (mu + sg * torch.randn_like(mu)).clamp(0, 1); p = torch.sigmoid(m[..., -1:])
        y = torch.cat([c, (torch.rand_like(p) < p).float() if hard_out else p], -1)
        pg = torch.sigmoid(hg); gate = (torch.rand_like(pg) < pg).float()
        if grad: gate = gate + pg - pg.detach()          # straight-through estimator
        if first or not hold: gate = torch.ones_like(gate)
        return gate * y + (1 - gate) * xprev
    def infer(s, x, tau):
        B = len(x); h, _ = s.enc(x)
        if s.use_u:
            um, ulv = s.qu(h.mean(1)).chunk(2, -1); ulv = ulv.clamp(-8, 3); u = um + torch.randn_like(um) * (.5 * ulv).exp()
            klu = .5 * (um ** 2 + ulv.exp() - ulv - 1).sum(-1)
        else: u = x.new_zeros(B, s.U); klu = x.new_zeros(B)
        sp = x.new_zeros(B, s.K); zp = x.new_zeros(B, s.Z); d = x.new_zeros(B)
        Zs, Ss, kls, klz, qps = [], [], [], [], []
        for t in range(T):
            ql = s.qs(torch.cat([h[:, t], sp, zp, u], -1)); pl = s.prior_s(sp, d, zp, u)
            st = F.gumbel_softmax(ql, tau=tau, hard=True); qp = F.softmax(ql, -1)
            kls.append((qp * (F.log_softmax(ql, -1) - F.log_softmax(pl, -1))).sum(-1))
            qd, qlv = s.qz(torch.cat([h[:, t], st, zp, u], -1)).chunk(2, -1); qm, qlv = torch.sigmoid(s.rho_logit) * zp + qd + st @ s.sm, qlv.clamp(-8, 3)
            pm, plv = s.prior_z(zp, st, u); z = qm + torch.randn_like(qm) * (.5 * qlv).exp()
            klz.append(.5 * (plv - qlv + (qlv.exp() + (qm - pm) ** 2) / plv.exp() - 1).sum(-1))
            same = (st * sp).sum(-1); d = same * (d + 1) + (1 - same); sp, zp = st, z
            Zs.append(z); Ss.append(st); qps.append(qp)
        return [torch.stack(a_, 1) for a_ in (Zs, Ss, kls, klz, qps)] + [klu, u]
    def generate(s, n, tau=.5, hard_out=True, grad=False):
        with torch.set_grad_enabled(grad):
            dev = next(s.parameters()).device; sp = torch.zeros(n, s.K, device=dev); zp = torch.zeros(n, s.Z, device=dev); d = torch.zeros(n, device=dev)
            u = torch.randn(n, s.U, device=dev) if s.use_u else torch.zeros(n, s.U, device=dev); xprev = torch.zeros(n, D, device=dev); outs = []
            for t in range(T):
                st = F.gumbel_softmax(s.prior_s(sp, d, zp, u), tau=tau, hard=True)
                pm, plv = s.prior_z(zp, st, u); z = pm + torch.randn_like(pm) * (.5 * plv).exp()
                m, ls, hg = s.decode(z, st, u); xprev = s.sample_step(m, ls, hg, xprev, t == 0, hard_out, grad, s.use_hold); outs.append(xprev)
                same = (st * sp).sum(-1); d = same * (d + 1) + (1 - same); sp, zp = st, z
            return torch.stack(outs, 1)

def mmd(a, b, scales=(.01, .03, .1, .3)):
    k = lambda x, y: sum(torch.exp(-(torch.cdist(x, y) ** 2 / x.shape[1]) / s) for s in scales)
    return k(a, a).mean() + k(b, b).mean() - 2 * k(a, b).mean()

def build_bank(Utr, Uva, n=2048, k=10):
    idx = np.random.default_rng(3).choice(len(Utr), min(n, len(Utr)), replace=False)
    M = torch.tensor(flat(Utr[idx]), device=DEV); FD = M.shape[1]
    dm = torch.cdist(M, M) / math.sqrt(FD); dm.fill_diagonal_(float("inf"))
    kd = dm.topk(min(k, len(M) - 1), largest=False).values[:, -1]; w = (kd / kd.mean()).clamp(.5, 2.)   # isolated trajectories -> larger margin
    V = torch.tensor(flat(Uva[:len(M)]), device=DEV); dv = (torch.cdist(V, M) / math.sqrt(FD)).min(1).values
    return dict(M=M, w=w, delta=torch.quantile(dv, .10).item())

def priv_pen(g, bank):
    dmin, idx = (torch.cdist(g, bank["M"]) / math.sqrt(g.shape[1])).min(1)
    return (F.relu(bank["delta"] * bank["w"][idx] - dmin) / bank["delta"]).mean()

def dyn_stats(x):
    """Per-feature lag-1/3/10 autocorrelation (after removing each sequence mean) and mean absolute hourly change, batch averaged."""
    xc = x - x.mean(1, keepdim=True); den = (xc ** 2).sum(1) + 1e-6
    ac = torch.stack([(xc[:, :-l] * xc[:, l:]).sum(1) / den for l in (1, 2, 3, 5, 8, 12, 18, 24)], 0).mean(1)
    return ac, (x[:, 1:] - x[:, :-1]).abs().mean((0, 1))

def marg_loss(xr, xg):
    """Per-feature 1-D Wasserstein-1 between pooled real and generated values (sorted-sample matching); stops saturation at the bounds."""
    a, b = xr.reshape(-1, D), xg.reshape(-1, D); sd = a.std(0).clamp_min(1e-3).detach()   # scale-free: narrow-range variables count as much as wide ones
    return ((a.sort(0).values - b.sort(0).values).abs().mean(0) / sd).mean()

def corr_mat(x):
    a = x.reshape(-1, D); a = a - a.mean(0); c = (a.T @ a) / (len(a) - 1); sd = c.diag().clamp_min(1e-6).sqrt(); return c / (sd[:, None] * sd[None, :])

def cor_loss(xr, xg): return (corr_mat(xr) - corr_mat(xg)).abs().mean()

def cw_stats(x):
    """Cross-window coupling: correlation across patients between first-half and second-half variable means."""
    a, b = x[:, :T // 2].mean(1), x[:, T // 2:].mean(1); a, b = a - a.mean(0), b - b.mean(0)
    return (a.T @ b) / (len(a) - 1) / (a.std(0).clamp_min(1e-4)[:, None] * b.std(0).clamp_min(1e-4)[None, :])

def cw_loss(xr, xg): return (cw_stats(xr) - cw_stats(xg)).abs().mean()

def dyn_loss(xr, xg):
    ar, dr = dyn_stats(xr); ag, dg_ = dyn_stats(xg); sd = xr.reshape(-1, D).std(0).clamp_min(1e-3).detach()
    return F.mse_loss(ag, ar) + (((dg_ - dr) / sd) ** 2).mean()   # hourly change in units of the variable's own spread, so saturating at a bound no longer pays

def train_clip(Utr, bank, epochs, rule=True, priv=True, mm=True, dwell=True, dyn=True, useu=True, marg=True, sw=True, cor=True, cw=True, hold=True, tag="CLiP-SSM", K=8):
    model = CLiP(K=K, use_dwell=dwell, use_u=useu, use_hold=hold).to(DEV); bs = CFG["BATCH"]; X = torch.tensor(Utr, device=DEV)
    opt = torch.optim.Adam(model.parameters(), 2e-3); steps = epochs * (len(X) // bs)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    for ep in range(epochs):
        fr = ep / epochs; beta = min(1., fr / .3); ramp = min(1., fr / .2); tau = max(.5, 1 - .5 * fr)
        p = torch.randperm(len(X), device=DEV); log = np.zeros(10); nb = 0
        for i in range(0, len(X) - bs + 1, bs):
            xb = X[p[i:i + bs]]; Zs, Ss, kls, klz, qp, klu, u = model.infer(xb, tau); m, ls, hg = model.decode(Zs, Ss, u)
            mu = torch.sigmoid(m[..., :-1]); sg = F.softplus(ls[..., :-1] - 3) + SIG_FLOOR
            gt = torch.ones_like(xb); gt[:, 1:] = ((xb[:, 1:] - xb[:, :-1]).abs() > 1e-6).float()      # 1 when a new value was recorded, 0 when it was held
            if not hold: gt = torch.ones_like(xb)
            nll = ((.5 * ((xb[..., :-1] - mu) / sg) ** 2 + torch.log(sg)) * gt[..., :-1]).sum(-1) + F.binary_cross_entropy_with_logits(m[..., -1], xb[..., -1], reduction="none") * gt[..., -1] \
                  + F.binary_cross_entropy_with_logits(hg, gt, reduction="none").sum(-1)
            pb = qp.mean((0, 1)); ent = -(pb * (pb + 1e-8).log()).sum()
            swc = 1 - (Ss[:, 1:] * Ss[:, :-1]).sum(-1).mean()   # fraction of hours with a regime change
            loss = nll.mean() + beta * ((kls + klz).mean() + klu.mean() / T) - CFG["LAM_ENT"] * ent + (CFG["LAM_SW"] * swc if sw else 0.); lr_, lp_, lm_, ld_, lw_, lc_, lx_ = 0., 0., 0., 0., 0., 0., 0.
            if rule or priv or mm or dyn or marg or cor or cw:
                g = model.generate(bs, tau, hard_out=False, grad=True)
                if rule: r = rule_hinge(g).sum(-1).mean(); loss = loss + ramp * CFG["LAM_RULE"] * r; lr_ = r.item()
                if priv: pv = priv_pen(g.reshape(bs, -1), bank); loss = loss + ramp * CFG["LAM_PRIV"] * pv; lp_ = pv.item()
                if mm: mv = mmd(xb.reshape(bs, -1), g.reshape(bs, -1)); loss = loss + ramp * CFG["LAM_MMD"] * mv; lm_ = mv.item()
                if marg: wl = marg_loss(xb, g); loss = loss + ramp * CFG["LAM_MARG"] * wl; lw_ = wl.item()
                if cor: cl_ = cor_loss(xb, g); loss = loss + ramp * CFG["LAM_COR"] * cl_; lc_ = cl_.item()
                if cw: xl_ = cw_loss(xb, g); loss = loss + ramp * CFG["LAM_CW"] * xl_; lx_ = xl_.item()
                if dyn: dl = dyn_loss(xb, g); loss = loss + ramp * CFG["LAM_DYN"] * dl; ld_ = dl.item()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.); opt.step(); sched.step()
            log += [nll.mean().item(), (kls + klz).mean().item(), lr_, lp_, lm_, ld_, lw_, swc.item(), lc_, lx_]; nb += 1
        if ep % max(1, epochs // 10) == 0: print(tag, "ep", ep, "nll/kl/rule/priv/mmd/dyn/marg/switch/cor/cw", np.round(log / nb, 4), "states used %.1f" % (1 / (pb ** 2).sum().item()))
    return model.eval()

## 5. Run everything (multiple seeds; ablations on the first seed)

In [ ]:
from hmmlearn.hmm import GaussianHMM
Utr, Uva = SPL["tr"], SPL["va"]; NS = CFG["N_SYN"]; SEEDS = CFG["SEEDS"]
results, SYN, MODELS = [], {}, {}
bank = build_bank(Utr, Uva); print("privacy margin delta=%.4f" % bank["delta"])

def timed(f):
    t0 = time.time(); o = f(); return o, time.time() - t0

def run(name, S, seed, t_train):
    r = evaluate(S, name=name); r.update(utility(S)); r["seed"] = seed; r["train_s"] = t_train; results.append(r)
    if seed == SEEDS[0]: SYN[name] = S
    print(f"[seed {seed}] {name}: KS {r['KS']:.4f} Corr {r['CorrPres']:.4f} Rule {r['RuleRec%']:.1f} MIA-res {r['MIA_resist%']:.1f} "
          f"UtilAUC {r['Util_auc%']:.1f} train {t_train:.0f}s")

def run_all(seed):
    torch.manual_seed(seed); np.random.seed(seed); first = seed == SEEDS[0]
    def fit_hmm():
        h = GaussianHMM(6, covariance_type="diag", n_iter=30, min_covar=1e-3, random_state=seed)
        h.fit(Utr.reshape(-1, D) + 1e-3 * np.random.rand(len(Utr) * T, D), lengths=[T] * len(Utr)); return h
    EB = CFG["EP_BASE"]
    hm, t = timed(fit_hmm); run("HMM", np.stack([hm.sample(T)[0] for _ in range(NS)]).astype(np.float32), seed, t)
    gfn, t = timed(lambda: fit_rgan(Utr, EB)); run("RGAN", batched(gfn, NS), seed, t)
    rv = RecVAE().to(DEV); _, t = timed(lambda: fit_generic(rv, rv.loss, Utr, EB, lr=BASE_LR['RecurrentVAE'])); run("RecurrentVAE", batched(rv.sample, NS), seed, t)
    gfn, t = timed(lambda: fit_timegan(Utr, EB)); run("TimeGAN", batched(gfn, NS), seed, t)
    tv = TimeVAE().to(DEV); _, t = timed(lambda: fit_generic(tv, tv.loss, Utr, EB, lr=BASE_LR['TimeVAE'])); run("TimeVAE", batched(tv.sample, NS), seed, t)
    mg, t = timed(lambda: fit_medgan(Utr, EB)); run("MedGAN-lite", batched(mg.sample, NS), seed, t)
    td = Diff(BRNNNet()).to(DEV); _, t = timed(lambda: fit_generic(td, td.loss, Utr, EB, lr=BASE_LR['TimeDiff'])); run("TimeDiff-lite", batched(td.sample, NS, 500), seed, t)
    dt_ = Diff(TFNet()).to(DEV); _, t = timed(lambda: fit_generic(dt_, dt_.loss, Utr, EB, lr=BASE_LR['DiffusionTS'])); run("DiffusionTS-lite", batched(dt_.sample, NS, 500), seed, t)
    dg = DGHMM().to(DEV); _, t = timed(lambda: fit_generic(dg, dg.loss, Utr, EB, lr=BASE_LR['DGHMM'])); run("DG-HMM (repro)", batched(dg.sample, NS), seed, t)
    cl, t = timed(lambda: train_clip(Utr, bank, CFG["EP_CLIP"])); run("CLiP-SSM (ours)", batched(lambda n: cl.generate(n), NS), seed, t)
    if first:
        MODELS.update({"dg": dg, "clip": cl})
    if CFG["RUN_ABLATIONS"] and seed in CFG["ABL_SEEDS"]:
        for nm, kw in [("CLiP -rule", dict(rule=False)), ("CLiP -priv", dict(priv=False)), ("CLiP -mmd", dict(mm=False)), ("CLiP -dyn", dict(dyn=False)),
                       ("CLiP -marg", dict(marg=False)), ("CLiP -cor", dict(cor=False)), ("CLiP -cw", dict(cw=False)), ("CLiP -hold", dict(hold=False))]:
            a, t = timed(lambda: train_clip(Utr, bank, CFG["EP_CLIP"], tag=nm, **kw)); run(nm, batched(lambda n: a.generate(n), NS), seed, t)

BASE_LR = dict(RecurrentVAE=2e-3, TimeVAE=2e-3, TimeDiff=1e-3, DiffusionTS=5e-4, DGHMM=2e-3)
def val_score(S):
    r = evaluate(S, split=dict(tr=Utr, va=Uva, te=Uva), name="v"); return r["KS"] + (1 - r["CorrPres"]) + r["ACFcurve_err"] + r["DTW"]
def tune_baselines():
    """Give each gradient-trained baseline the same kind of validation-based tuning as CLiP-SSM: three learning rates, 100 epochs, pick by validation score."""
    mk = dict(RecurrentVAE=lambda: RecVAE(), TimeVAE=lambda: TimeVAE(), TimeDiff=lambda: Diff(BRNNNet()), DiffusionTS=lambda: Diff(TFNet()), DGHMM=lambda: DGHMM())
    for nm, build in mk.items():
        best = None
        for lr in [BASE_LR[nm] * 3, BASE_LR[nm], BASE_LR[nm] / 3]:
            torch.manual_seed(0); m = build().to(DEV); fit_generic(m, m.loss, Utr, max(2, CFG["EP_BASE"] // 2), lr=lr)
            sc = val_score(batched(m.sample, min(NS, 2000), 500)); print("tune", nm, "lr %.5f" % lr, "val score %.4f" % sc)
            if best is None or sc < best[0]: best = (sc, lr)
        BASE_LR[nm] = best[1]; print("selected", nm, "lr", best[1])
if CFG["TUNE_BASE"]: tune_baselines()

def tune():
    """Pick LAM_MMD and LAM_DYN on the VALIDATION set (the test set is never used for selection)."""
    EVS = dict(tr=Utr, va=Uva, te=Uva); best = None
    for lm, ld in [(30., 30.), (80., 30.), (30., 100.), (80., 100.)]:
        CFG["LAM_MMD"], CFG["LAM_DYN"] = lm, ld
        m_ = train_clip(Utr, bank, max(2, CFG["EP_CLIP"] // 2), tag=f"tune mmd{lm:.0f} dyn{ld:.0f}")
        r = evaluate(batched(lambda n: m_.generate(n), min(NS, 2000)), split=EVS, name="tune")
        sc = r["KS"] + (1 - r["CorrPres"]) + r["ACFcurve_err"] + r["DTW"]
        print("tune", lm, ld, "val score %.4f" % sc, {k: round(r[k], 3) for k in ["KS", "CorrPres", "ACFcurve_err", "DTW"]})
        if best is None or sc < best[0]: best = (sc, lm, ld)
    CFG["LAM_MMD"], CFG["LAM_DYN"] = best[1], best[2]; print("selected LAM_MMD=%.0f LAM_DYN=%.0f" % (best[1], best[2]))
if CFG["TUNE"]: tune()

for sd in SEEDS: run_all(sd)

## 6. Tables (mean +/- std over seeds), win/loss, significance, LaTeX

In [ ]:
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
RA = pd.DataFrame(results); RA.to_csv(f"{OUT}/results_all_{MODE}.csv", index=False)
cols = [c for c in RA.columns if c not in ("model", "seed")]
mean, std = RA.groupby("model", sort=False)[cols].mean(), RA.groupby("model", sort=False)[cols].std().fillna(0)
tab = mean.round(4).astype(str) + " +/- " + std.round(4).astype(str)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50); print(tab.T)
tab.to_csv(f"{OUT}/table_mean_std_{MODE}.csv")
try: open(f"{OUT}/table_{MODE}.tex", "w").write(tab.T.to_latex())
except Exception as e: print("latex export skipped:", e)
better_low = ["KS", "ACF10_err", "ACFcurve_err", "DTW", "SkewErr", "KurtErr", "RuleGap", "MIA_cal_dev", "MIA_worstgrp_AUC"]
better_high = ["CorrPres", "Bounds%_raw", "MIA_resist%", "Util_fore%", "Util_auc%"]
ours = "CLiP-SSM (ours)"; rest = [m for m in mean.index if not m.startswith("CLiP")]
print("\nCLiP-SSM vs best other method (means):")
for k in better_low + better_high:
    best = mean.loc[rest, k].min() if k in better_low else mean.loc[rest, k].max(); bm = (mean.loc[rest, k].idxmin() if k in better_low else mean.loc[rest, k].idxmax())
    win = mean.loc[ours, k] < best if k in better_low else mean.loc[ours, k] > best
    line = f"{k:18s} ours={mean.loc[ours, k]:9.4f}  best_other={best:9.4f} ({bm})  {'WIN' if win else 'loss'}"
    if len(SEEDS) >= 5:
        a = RA[RA.model == ours].sort_values("seed")[k].values; b = RA[RA.model == bm].sort_values("seed")[k].values
        try: line += f"  Wilcoxon p={stats.wilcoxon(a, b).pvalue:.4f}"
        except Exception: pass
    print(line)
if len(SEEDS) < 5: print("(<5 seeds: no significance test. Set CFG['SEEDS']=[0,1,2,3,4] for the final paper run.)")
print("\nPaper's reported DG-HMM (different preprocessing, not reproducible 1:1): KS 0.014, Corr 0.942, ACF@10 0.876, DTW 0.89, rule 96.3%, bounds 94.8%, MIA-resist 89.4%, utility loss 2-5%")
print("\nReal-data reference (val vs test):", {k: round(v, 4) for k, v in real_ref.items() if k != "model"})
np.savez_compressed(f"{OUT}/real_test_{MODE}.npz", te=SPL["te"][:1000])
np.savez_compressed(f"{OUT}/synthetic_{MODE}.npz", **{k.replace(" ", "_"): v[:1000] for k, v in SYN.items()})

## 7. Figures for the paper (saved as 600-dpi PNG)
Fig1 metric comparison | Fig2 trajectories | Fig3 correlation error | Fig4 ACF | Fig5 privacy (DCR + subgroup MIA) | Fig6 state-duration analysis | Fig7 ablation | Fig8 sensitivity (if RUN_SWEEP) | Fig9 cost

In [ ]:
import matplotlib.pyplot as plt
PAPER_NAME = {"fig1_metrics": "fig_metrics", "fig2_trajectories": "fig_traj", "fig4_acf": "fig_acf", "fig5_privacy": "fig_privacy", "fig6_dwell": "fig_dwell", "fig7_ablation": "fig_ablation"}
def sv(n):
    plt.tight_layout(); plt.savefig(f"{OUT}/{n}_{MODE}.png", dpi=600)
    if n in PAPER_NAME: plt.savefig(f"{OUT}/{PAPER_NAME[n]}.png", dpi=600)   # file names used by the LaTeX source
    plt.show()
main = [m for m in mean.index if not m.startswith("CLiP -")]; col = plt.cm.tab10(np.arange(len(main)))
units = lambda U: LO + U * (HI - LO)
# Fig1
mets = ["KS", "CorrPres", "ACFcurve_err", "DTW", "RuleGap", "MIA_cal_dev", "Util_fore%", "Util_auc%"]
fig, ax = plt.subplots(2, 4, figsize=(18, 7))
for a, k in zip(ax.ravel(), mets):
    a.bar(range(len(main)), mean.loc[main, k], yerr=std.loc[main, k], color=col); a.set_title(k + (" (lower better)" if k in better_low else " (higher better)"), fontsize=9)
    a.set_xticks(range(len(main))); a.set_xticklabels(main, rotation=60, ha="right", fontsize=7)
sv("fig1_metrics")
# Fig2 trajectories
R = SPL["te"]; show = {"Real (test)": R, "DG-HMM (repro)": SYN["DG-HMM (repro)"], "CLiP-SSM": SYN[ours]}
fk = ["hr", "sbp", "spo2", "glucose"]; fig, ax = plt.subplots(3, 4, figsize=(18, 8))
for i, (nm, S) in enumerate(show.items()):
    S = np.clip(S, 0, 1)
    for j, f in enumerate(fk):
        x = units(S)[:, :, IDX[f]]; ax[i, j].plot(x[:8].T, alpha=.35, lw=.8); ax[i, j].plot(x.mean(0), "k", lw=2)
        ax[i, j].set_title(f"{nm} - {f}", fontsize=9)
sv("fig2_trajectories")
# Fig3 correlation error
cr = lambda S: np.nan_to_num(np.corrcoef(np.clip(S, 0, 1).reshape(-1, D).T)); C0 = cr(R); nm_ = [f[0] for f in FEATS]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for a, (t_, M) in zip(ax, [("Real corr", C0), ("|dC| DG-HMM", np.abs(C0 - cr(SYN["DG-HMM (repro)"]))), ("|dC| CLiP-SSM", np.abs(C0 - cr(SYN[ours])))]):
    im = a.imshow(M, cmap="coolwarm" if t_ == "Real corr" else "viridis", vmin=-1 if t_ == "Real corr" else 0, vmax=1 if t_ == "Real corr" else .5); a.set_title(t_)
    a.set_xticks(range(D)); a.set_xticklabels(nm_, rotation=90); a.set_yticks(range(D)); a.set_yticklabels(nm_); plt.colorbar(im, ax=a)
sv("fig3_corr")
# Fig4 ACF
lags = range(1, 25); plt.figure(figsize=(7, 4.5)); plt.plot(lags, [np.nanmean(acf_lag(R, l)) for l in lags], "k", lw=3, label="Real")
for m in ["HMM", "RecurrentVAE", "TimeGAN", "TimeVAE", "TimeDiff-lite", "DiffusionTS-lite", "DG-HMM (repro)", ours]:
    S = np.clip(SYN[m], 0, 1); plt.plot(lags, [np.nanmean(acf_lag(S, l)) for l in lags], label=m)
plt.xlabel("lag (h)"); plt.ylabel("mean ACF"); plt.legend(fontsize=7); sv("fig4_acf")
# Fig5 privacy
def mia_groups(S_raw):
    S = np.clip(S_raw, 0, 1).astype(np.float32); S[..., -1] = (S[..., -1] > .5); n = min(2000, len(SPL["te"]), len(SPL["tr"]))
    mem, non = SPL["tr"][:n], SPL["te"][:n]; sm, sn = -min_dist(mem, S), -min_dist(non, S); gm, gn = groups(mem), groups(non)
    o = {"all": roc_auc_score(np.r_[np.ones(n), np.zeros(n)], np.r_[sm, sn])}
    for k in gm:
        if gm[k].sum() > 20 and gn[k].sum() > 20: o[k] = roc_auc_score(np.r_[np.ones(gm[k].sum()), np.zeros(gn[k].sum())], np.r_[sm[gm[k]], sn[gn[k]]])
    return o
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5)); dt = min_dist(R[:2000], Utr)
for m in ["DG-HMM (repro)", ours]:
    S = np.clip(SYN[m], 0, 1); S[..., -1] = S[..., -1] > .5; ax[0].hist(min_dist(S[:2000], Utr), 50, alpha=.5, label=m + " -> train")
ax[0].hist(dt, 50, alpha=.5, label="test -> train", histtype="step", color="k", lw=2); ax[0].legend(fontsize=7); ax[0].set_title("Distance to closest training record")
G = {m: mia_groups(SYN[m]) for m in main}; gk = list(G[ours].keys()); w = .8 / len(main)
for i, m in enumerate(main): ax[1].bar(np.arange(len(gk)) + i * w, [G[m].get(k, np.nan) for k in gk], w, label=m, color=col[i])
ax[1].axhline(.5, c="k", ls="--"); ax[1].set_xticks(np.arange(len(gk)) + .4); ax[1].set_xticklabels(gk); ax[1].set_ylabel("MIA AUC (0.5 = safe)"); ax[1].set_ylim(.4, None); ax[1].legend(fontsize=6)
sv("fig5_privacy"); print(pd.DataFrame(G).round(3))
# Fig6 state-duration analysis
def runlens(Sx):
    out = []
    for row in Sx:
        c = 1
        for a, b in zip(row[:-1], row[1:]):
            if a == b: c += 1
            else: out.append(c); c = 1
        out.append(c)
    return np.array(out)
cl, dg = MODELS["clip"], MODELS["dg"]
def clip_states(U):
    with torch.no_grad(): return np.concatenate([cl.infer(torch.tensor(U[i:i + 500], device=DEV), .5)[1].argmax(-1).cpu().numpy() for i in range(0, len(U), 500)])
rl_real, rl_gen = runlens(clip_states(R)), runlens(clip_states(np.clip(SYN[ours], 0, 1).astype(np.float32)))
with torch.no_grad():
    A = F.softmax(dg.A, -1).cpu().numpy(); pi = F.softmax(dg.pi, 0).cpu().numpy(); r_ = np.random.default_rng(0); paths = []
    for _ in range(2000):
        s_ = r_.choice(len(pi), p=pi); p_ = [s_]
        for _t in range(T - 1): s_ = r_.choice(len(pi), p=A[s_]); p_.append(s_)
        paths.append(p_)
rl_dg = runlens(np.array(paths)); bins = np.arange(1, 25)
plt.figure(figsize=(7, 4.5))
for rl, nm in [(rl_real, "Real (CLiP posterior)"), (rl_gen, "CLiP-SSM generated"), (rl_dg, "DG-HMM state chain")]: plt.hist(rl, bins, density=True, alpha=.5, label=nm)
plt.xlabel("state dwell time (h)"); plt.legend(); plt.title("State duration: semi-Markov vs geometric"); sv("fig6_dwell")
print("mean dwell: real %.2f | CLiP %.2f | DG-HMM chain %.2f" % (rl_real.mean(), rl_gen.mean(), rl_dg.mean()))
# Fig7 ablation
ab = [m for m in mean.index if m.startswith("CLiP")]
if len(ab) > 1:
    fig, ax = plt.subplots(1, 5, figsize=(20, 4))
    for a, k in zip(ax, ["KS", "CorrPres", "RuleRec%", "MIA_resist%", "DCR_p5_ratio"]):
        a.bar(range(len(ab)), mean.loc[ab, k], color=plt.cm.Set2(np.arange(len(ab)))); a.set_xticks(range(len(ab))); a.set_xticklabels(ab, rotation=45, ha="right", fontsize=8); a.set_title(k)
    sv("fig7_ablation")
# Fig8 sensitivity
if CFG["RUN_SWEEP"]:
    sw = []
    for lam in [0, 5, 20, 60]:
        old = CFG["LAM_PRIV"]; CFG["LAM_PRIV"] = lam; m_ = train_clip(Utr, bank, CFG["EP_CLIP"], priv=lam > 0, tag=f"lam{lam}"); CFG["LAM_PRIV"] = old
        r = evaluate(batched(lambda n: m_.generate(n), NS), name=f"lam_priv={lam}"); r["sweep"] = "lam"; r["val"] = lam; sw.append(r)
    for K in [4, 6, 8, 12]:
        m_ = train_clip(Utr, bank, CFG["EP_CLIP"], K=K, tag=f"K{K}"); r = evaluate(batched(lambda n: m_.generate(n), NS), name=f"K={K}"); r["sweep"] = "K"; r["val"] = K; sw.append(r)
    SW = pd.DataFrame(sw); SW.to_csv(f"{OUT}/sweep_{MODE}.csv", index=False)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4)); a_ = SW[SW.sweep == "lam"]; ax[0].plot(a_.MIA_AUC, a_.KS, "o-")
    for _, r in a_.iterrows(): ax[0].annotate(f"lam={int(r.val)}", (r.MIA_AUC, r.KS))
    ax[0].set_xlabel("MIA AUC"); ax[0].set_ylabel("KS"); ax[0].set_title("Privacy-fidelity trade-off")
    b_ = SW[SW.sweep == "K"]; ax[1].plot(b_.val, b_.CorrPres, "o-", label="CorrPres"); ax[1].plot(b_.val, 1 - b_.KS, "s-", label="1-KS"); ax[1].set_xlabel("K states"); ax[1].legend()
    sv("fig8_sensitivity")
# Fig9 cost
cost = RA.groupby("model", sort=False).train_s.mean(); print("\nTraining time (s):\n", cost.round(0))
t0 = time.time(); _ = batched(lambda n: cl.generate(n), 1000); print("CLiP generation of 1000 sequences: %.1fs; params: %d" % (time.time() - t0, sum(p.numel() for p in cl.parameters())))
torch.save(cl.state_dict(), f"{OUT}/clip_ssm_{MODE}.pt")

## 8. Cross-dataset summary (run the notebook once per dataset, keep the `results_all_*.csv` files in one folder, then run this cell)

In [ ]:
import glob
files = sorted(glob.glob(f"{OUT}/results_all_*.csv") + glob.glob("/kaggle/input/*/results_all_*.csv"))
files = [f for f in files if "demo" not in f]
if len(files) < 2: print("Only", len(files), "real dataset result file(s) found; run the other datasets, then re-run this cell.")
rows = []
for f in files:
    ds = os.path.basename(f)[len("results_all_"):-4]; r = pd.read_csv(f); mu = r.groupby("model").mean(numeric_only=True)
    oth = [m for m in mu.index if not m.startswith("CLiP")]
    for k in better_low + better_high:
        b = mu.loc[oth, k].min() if k in better_low else mu.loc[oth, k].max()
        rows.append(dict(dataset=ds, metric=k, ours=mu.loc[ours, k], best_other=b, best_method=(mu.loc[oth, k].idxmin() if k in better_low else mu.loc[oth, k].idxmax()),
                         win=(mu.loc[ours, k] < b) if k in better_low else (mu.loc[ours, k] > b)))
if rows:
    X_ = pd.DataFrame(rows); X_.to_csv(f"{OUT}/cross_dataset_summary.csv", index=False)
    print(X_.pivot(index="metric", columns="dataset", values="win")); print("\nwin-rate per dataset:\n", X_.groupby("dataset").win.mean().round(2))

## 9. Which metrics did training target? (report both groups in the paper)

In [ ]:
targeted = ["KS", "CorrPres", "ACFcurve_err", "SkewErr"]            # directly shaped by the marginal, correlation, dynamics and MMD losses
heldout = [k for k in better_low + better_high if k not in targeted]  # not optimised by any loss term
for nm, grp in [("TARGETED by training losses", targeted), ("HELD-OUT (not optimised)", heldout)]:
    w = 0
    for k in grp:
        o = mean.loc[rest, k]; best = o.min() if k in better_low else o.max()
        w += (mean.loc[ours, k] < best) if k in better_low else (mean.loc[ours, k] > best)
    print(f"{nm}: CLiP-SSM best on {int(w)} of {len(grp)}  ->  {grp}")